In [ ]:
"""
BSTabDiff component/hyperparameter sensitivity analysis on Colon.

Design
------
* Two tuned variants:
    1) BSTabDiff without GO-BS.
    2) BSTabDiff with GO-BS.
* M is fixed at the tuned Colon value M=8 because M is evaluated separately.
* One-factor-at-a-time sensitivity around each tuned base configuration.
* Logistic Regression only.
* Strict fold-wise 5x5 repeated stratified CV.
* Reports TSTR, TRTR, C2ST, runtime, and peak GPU memory.
* Physical GPU ID 4 is isolated with CUDA_VISIBLE_DEVICES=4. Inside the
  process/notebook it is addressed as cuda:0.
"""

# Must run before importing torch.
import os
import sys
import warnings

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # physical GPU ID 4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONWARNINGS"] = "ignore"

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import gc
import inspect
import json
import random
import time
import traceback
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

try:
    import torch
except Exception as exc:
    raise ImportError("PyTorch is required for BSTabDiff.") from exc

from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# =============================================================================
# User configuration
# =============================================================================
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
OUTPUT_DIR = Path("colon_component_sensitivity_cuda4")

GLOBAL_SEED = 42
CV_SEED = 0
N_SPLITS = 5
N_REPEATS = 5
RESUME = True

# M is fixed because a separate experiment already studies M.
FIXED_M = 8
BASE_N_SYN = 200
TORCH_DEVICE_STR = "cuda:0"  # physical GPU 4 after isolation

# If your implementation uses different keyword names for the two conditional
# mechanisms, add them to the alias lists below. The code auto-detects the
# first compatible name in fit_block_subunit_generator's signature.
OPTIONAL_FLAG_ALIASES = {
    "class_conditional_marginals": (
        "class_conditional_marginals",
        "use_class_conditional_marginals",
        "class_cond_marginals",
        "use_cc_marginals",
        "conditional_marginals",
        "condition_marginals_on_y",
        "marginals_condition_on_y",
    ),
    "class_conditional_missingness": (
        "class_conditional_missingness",
        "use_class_conditional_missingness",
        "class_cond_missingness",
        "use_cc_missingness",
        "conditional_missingness",
        "condition_missingness_on_y",
        "missingness_condition_on_y",
    ),
}


# =============================================================================
# Fixed tuned configurations
# =============================================================================
# Best no-GO-BS Colon parameters from the 2,000-trial study.
BEST_NO_GOBS: Dict[str, object] = {
    "M": FIXED_M,
    "prior_type": "flow",
    "prior_epochs": 300,
    "prior_batch": 64,
    "prior_lr": 0.0032931446587915556,
    "use_ema": True,
    "ema_decay": 0.9925524818123974,
    "permute_features": False,
    "use_gobs": False,
    # API-compatible placeholders, unused when GO-BS is disabled.
    "gobs_num_clusters": 7,
    "gobs_metric": "correlation",
    "gobs_bins": 32,
    "gobs_top_k": None,
    "gobs_refine_order": False,
    "gobs_direction_select": False,
    "gobs_refine_passes": 0,
    "gobs_boundary_refine_passes": 1,
    "gobs_boundary_window": 8,
    "gobs_lambda_cross": 1.0,
    "gobs_gamma_balance": 1e-6,
}

# Same tuned generator parameters with the fixed best GO-BS front-end.
BEST_WITH_GOBS: Dict[str, object] = {
    "M": FIXED_M,
    "prior_type": "flow",
    "prior_epochs": 300,
    "prior_batch": 64,
    "prior_lr": 0.0032931446587915556,
    "use_ema": True,
    "ema_decay": 0.9925524818123974,
    "permute_features": False,
    "use_gobs": True,
    "gobs_num_clusters": 3,
    "gobs_metric": "correlation",
    "gobs_bins": 24,
    "gobs_top_k": 16,
    "gobs_refine_order": False,
    "gobs_direction_select": False,
    "gobs_refine_passes": 3,
    "gobs_boundary_refine_passes": 5,
    "gobs_boundary_window": 16,
    "gobs_lambda_cross": 2.3988509383361634,
    "gobs_gamma_balance": 5.193972052958215e-06,
}

VARIANTS: Tuple[Tuple[str, Dict[str, object]], ...] = (
    ("Tuned-no-GO-BS", BEST_NO_GOBS),
    ("Tuned-GO-BS", BEST_WITH_GOBS),
)

# A compatible high-performing diffusion configuration observed in the Colon
# Optuna trials. This is intentionally labeled a matched prior-family control,
# not a one-parameter-only perturbation, because flow and diffusion use very
# different stable learning-rate/epoch scales.
MATCHED_DIFFUSION_OVERRIDES = {
    "prior_type": "diffusion",
    "prior_epochs": 1200,
    "prior_batch": 128,
    "prior_lr": 7.04809e-05,
    "use_ema": True,
    "ema_decay": 0.99977,
}


# =============================================================================
# Sensitivity cases
# =============================================================================
def make_cases() -> List[Dict[str, object]]:
    base_lr = float(BEST_NO_GOBS["prior_lr"])
    return [
        {
            "case_id": "BASE",
            "factor": "baseline",
            "level": "tuned",
            "overrides": {},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "Prior-Diffusion-Matched",
            "factor": "prior_family",
            "level": "diffusion-matched",
            "overrides": dict(MATCHED_DIFFUSION_OVERRIDES),
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "Epochs-150",
            "factor": "prior_epochs",
            "level": "150",
            "overrides": {"prior_epochs": 150},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "Epochs-800",
            "factor": "prior_epochs",
            "level": "800",
            "overrides": {"prior_epochs": 800},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "Epochs-1500",
            "factor": "prior_epochs",
            "level": "1500",
            "overrides": {"prior_epochs": 1500},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "LR-0.5x",
            "factor": "prior_lr",
            "level": "0.5x",
            "overrides": {"prior_lr": 0.5 * base_lr},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "LR-2x",
            "factor": "prior_lr",
            "level": "2x",
            "overrides": {"prior_lr": 2.0 * base_lr},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "No-EMA",
            "factor": "ema",
            "level": "off",
            "overrides": {"use_ema": False},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "No-CC-Marginals",
            "factor": "class_conditional_marginals",
            "level": "off",
            "overrides": {"class_conditional_marginals": False},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "No-CC-Missingness",
            "factor": "class_conditional_missingness",
            "level": "off",
            "overrides": {"class_conditional_missingness": False},
            "n_syn": BASE_N_SYN,
        },
        {
            "case_id": "nSyn-100",
            "factor": "synthetic_budget",
            "level": "100",
            "overrides": {},
            "n_syn": 100,
        },
        {
            "case_id": "nSyn-500",
            "factor": "synthetic_budget",
            "level": "500",
            "overrides": {},
            "n_syn": 500,
        },
    ]


CASES = make_cases()


# =============================================================================
# Helpers
# =============================================================================
def log(message: str) -> None:
    print(message, flush=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_torch() -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass


def validate_device() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. Restart the notebook kernel and run the GPU setup "
            "cell before importing torch."
        )
    device = torch.device(TORCH_DEVICE_STR)
    torch.cuda.set_device(device)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
    return device


def make_lr(seed: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "classifier",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    random_state=seed,
                ),
            ),
        ]
    )


def predict_positive_probability(model: Pipeline, X: np.ndarray) -> np.ndarray:
    probabilities = model.predict_proba(X)
    if probabilities.ndim != 2 or probabilities.shape[1] < 2:
        raise RuntimeError("Expected binary predict_proba output with two columns.")
    return np.asarray(probabilities[:, 1], dtype=float)


def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "auc": float(roc_auc_score(y_true, probabilities)),
        "acc": float(accuracy_score(y_true, predictions)),
    }


def load_colon_dataset(csv_path: str, label_col: str) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    path = Path(csv_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Colon CSV not found at '{path.resolve()}'. Update DATA_FILE."
        )

    frame = pd.read_csv(path)
    if label_col not in frame.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. First columns: {list(frame.columns[:10])}"
        )

    y_raw = frame[label_col].to_numpy()
    X_frame = frame.drop(columns=[label_col]).select_dtypes(include=[np.number])
    X = X_frame.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)

    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary Colon labels, found {np.unique(y)}.")
    return X, y, list(X_frame.columns)


# =============================================================================
# Optional API flag resolution
# =============================================================================
FIT_SIGNATURE = inspect.signature(fit_block_subunit_generator)
FIT_PARAMETERS = FIT_SIGNATURE.parameters
HAS_VAR_KW = any(
    parameter.kind == inspect.Parameter.VAR_KEYWORD
    for parameter in FIT_PARAMETERS.values()
)


def resolve_optional_flag_name(logical_name: str) -> Optional[str]:
    for alias in OPTIONAL_FLAG_ALIASES[logical_name]:
        if alias in FIT_PARAMETERS:
            return alias
    if HAS_VAR_KW:
        # Canonical first alias; change the list if your **kwargs implementation
        # expects another spelling.
        return OPTIONAL_FLAG_ALIASES[logical_name][0]
    return None


RESOLVED_OPTIONAL_FLAGS = {
    name: resolve_optional_flag_name(name) for name in OPTIONAL_FLAG_ALIASES
}


class UnsupportedOptionalFlag(RuntimeError):
    pass


def materialize_case_params(
    base_params: Dict[str, object],
    case: Dict[str, object],
) -> Tuple[Dict[str, object], Dict[str, bool]]:
    params = dict(base_params)
    optional_flags: Dict[str, bool] = {}
    for key, value in dict(case["overrides"]).items():
        if key in OPTIONAL_FLAG_ALIASES:
            optional_flags[key] = bool(value)
        else:
            params[key] = value
    params["M"] = FIXED_M
    return params, optional_flags


# =============================================================================
# Generator fitting
# =============================================================================
def fit_generator(
    X_train: np.ndarray,
    y_train: np.ndarray,
    params: Dict[str, object],
    optional_flags: Dict[str, bool],
    seed: int,
    save_name: str,
):
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=int)
    feature_specs = [
        FeatureSpec(name=f"f{j}", kind="continuous")
        for j in range(X_train.shape[1])
    ]

    kwargs = dict(
        X=X_train,
        feature_specs=feature_specs,
        y=y_train,
        M=int(params["M"]),
        blocks=None,
        permute_features=bool(params["permute_features"]),
        prior_type=str(params["prior_type"]),
        device=TORCH_DEVICE_STR,
        seed=int(seed),
        prior_epochs=int(params["prior_epochs"]),
        prior_batch=int(params["prior_batch"]),
        prior_lr=float(params["prior_lr"]),
        verbose_every=0,
        save_dir=None,
        save_name=save_name,
        save_best=True,
        use_ema=bool(params["use_ema"]),
        ema_decay=float(params["ema_decay"]),
        return_train_info=False,
        use_gobs=bool(params["use_gobs"]),
        gobs_num_clusters=int(params["gobs_num_clusters"]),
        gobs_metric=str(params["gobs_metric"]),
        gobs_bins=int(params["gobs_bins"]),
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=bool(params["gobs_refine_order"]),
        gobs_direction_select=bool(params["gobs_direction_select"]),
        gobs_refine_passes=int(params["gobs_refine_passes"]),
        gobs_boundary_refine_passes=int(params["gobs_boundary_refine_passes"]),
        gobs_boundary_window=int(params["gobs_boundary_window"]),
        gobs_lambda_cross=float(params["gobs_lambda_cross"]),
        gobs_gamma_balance=float(params["gobs_gamma_balance"]),
        gobs_use_cpu_kmeans=False,
    )

    for logical_name, value in optional_flags.items():
        actual_name = RESOLVED_OPTIONAL_FLAGS[logical_name]
        if actual_name is None:
            raise UnsupportedOptionalFlag(
                f"Could not find a keyword for '{logical_name}' in "
                f"fit_block_subunit_generator{FIT_SIGNATURE}. Add your implementation's "
                f"keyword to OPTIONAL_FLAG_ALIASES."
            )
        kwargs[actual_name] = bool(value)

    return fit_block_subunit_generator(**kwargs)


def sanitize_synthetic_training_data(
    X_syn: np.ndarray,
    y_syn: np.ndarray,
    X_real_train: np.ndarray,
    y_real_train: np.ndarray,
    rng: np.random.RandomState,
) -> Tuple[np.ndarray, np.ndarray]:
    X_syn = np.nan_to_num(
        np.asarray(X_syn, dtype=np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    y_syn = np.asarray(y_syn, dtype=int)

    valid_classes = np.unique(y_real_train)
    valid_mask = np.isin(y_syn, valid_classes)
    X_syn = X_syn[valid_mask]
    y_syn = y_syn[valid_mask]

    missing_classes = [c for c in valid_classes if c not in np.unique(y_syn)]
    if missing_classes:
        additions_X: List[np.ndarray] = []
        additions_y: List[int] = []
        for class_value in missing_classes:
            candidates = np.where(y_real_train == class_value)[0]
            if len(candidates) == 0:
                continue
            chosen = int(candidates[rng.randint(0, len(candidates))])
            additions_X.append(X_real_train[chosen])
            additions_y.append(int(y_real_train[chosen]))
        if additions_X:
            X_syn = np.vstack([X_syn, np.asarray(additions_X, dtype=np.float32)])
            y_syn = np.concatenate([y_syn, np.asarray(additions_y, dtype=int)])

    if len(np.unique(y_syn)) < 2:
        raise RuntimeError("Synthetic training data contains fewer than two classes.")
    return X_syn, y_syn


# =============================================================================
# Evaluation
# =============================================================================
def evaluate_trtr_once(
    X: np.ndarray,
    y: np.ndarray,
    splits: Iterable[Tuple[np.ndarray, np.ndarray]],
    fold_seeds: Iterable[int],
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for fold_id, ((train_idx, test_idx), fold_seed) in enumerate(
        zip(splits, fold_seeds), start=1
    ):
        model = make_lr(int(fold_seed))
        model.fit(X[train_idx], y[train_idx])
        probabilities = predict_positive_probability(model, X[test_idx])
        metrics = binary_metrics(y[test_idx], probabilities)
        rows.append(
            {
                "fold": fold_id,
                "fold_seed": int(fold_seed),
                "trtr_acc": metrics["acc"],
                "trtr_auc": metrics["auc"],
            }
        )
    return pd.DataFrame(rows)


def evaluate_c2st(X_real_train: np.ndarray, X_syn: np.ndarray, seed: int) -> Dict[str, float]:
    rng = np.random.RandomState(seed)
    n_mix = min(len(X_real_train), len(X_syn))
    real_idx = rng.choice(len(X_real_train), size=n_mix, replace=False)
    syn_idx = rng.choice(len(X_syn), size=n_mix, replace=False)
    X_mix = np.vstack([X_real_train[real_idx], X_syn[syn_idx]])
    y_mix = np.concatenate([np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)])

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(splitter.split(X_mix, y_mix))
    model = make_lr(seed)
    model.fit(X_mix[train_idx], y_mix[train_idx])
    probabilities = predict_positive_probability(model, X_mix[test_idx])
    return binary_metrics(y_mix[test_idx], probabilities)


def load_existing_results(path: Path) -> pd.DataFrame:
    if RESUME and path.exists():
        frame = pd.read_csv(path)
        log(f"Resuming from {path} with {len(frame)} saved rows.")
        return frame
    return pd.DataFrame()


def result_already_complete(
    existing: pd.DataFrame,
    variant: str,
    case_id: str,
    fold_id: int,
) -> bool:
    if existing.empty:
        return False
    required = {"variant", "case_id", "fold", "status"}
    if not required.issubset(existing.columns):
        return False
    match = existing[
        (existing["variant"] == variant)
        & (existing["case_id"] == case_id)
        & (existing["fold"].astype(int) == int(fold_id))
        & (existing["status"] == "COMPLETE")
    ]
    return len(match) > 0


def append_and_save_result(
    current: pd.DataFrame,
    row: Dict[str, object],
    path: Path,
) -> pd.DataFrame:
    if not current.empty and {"variant", "case_id", "fold"}.issubset(current.columns):
        keep = ~(
            (current["variant"] == row["variant"])
            & (current["case_id"] == row["case_id"])
            & (current["fold"].astype(int) == int(row["fold"]))
        )
        current = current.loc[keep].copy()
    current = pd.concat([current, pd.DataFrame([row])], ignore_index=True)
    current = current.sort_values(["variant", "case_id", "fold"]).reset_index(drop=True)
    current.to_csv(path, index=False)
    return current


def run_sensitivity(
    X: np.ndarray,
    y: np.ndarray,
    splits: List[Tuple[np.ndarray, np.ndarray]],
    fold_seeds: np.ndarray,
    trtr_by_fold: pd.DataFrame,
    fold_results_path: Path,
) -> pd.DataFrame:
    results = load_existing_results(fold_results_path)
    trtr_map = trtr_by_fold.set_index("fold").to_dict(orient="index")
    total_runs = len(VARIANTS) * len(CASES) * len(splits)
    counter = 0

    for variant_name, base_params in VARIANTS:
        for case in CASES:
            params, optional_flags = materialize_case_params(base_params, case)
            case_id = str(case["case_id"])
            n_syn = int(case["n_syn"])

            log("\n" + "=" * 100)
            log(
                f"Variant={variant_name} | Case={case_id} | "
                f"factor={case['factor']} | level={case['level']}"
            )
            log("=" * 100)

            for fold_id, ((train_idx, test_idx), fold_seed) in enumerate(
                zip(splits, fold_seeds), start=1
            ):
                if result_already_complete(results, variant_name, case_id, fold_id):
                    counter += 1
                    log(
                        f"[Skip {counter:03d}/{total_runs}] {variant_name}, "
                        f"{case_id}, fold={fold_id:02d}"
                    )
                    continue

                X_train, y_train = X[train_idx], y[train_idx]
                X_test, y_test = X[test_idx], y[test_idx]
                fold_rng = np.random.RandomState(int(fold_seed))

                row: Dict[str, object] = {
                    "variant": variant_name,
                    "use_gobs": bool(params["use_gobs"]),
                    "case_id": case_id,
                    "factor": str(case["factor"]),
                    "level": str(case["level"]),
                    "M": FIXED_M,
                    "prior_type": str(params["prior_type"]),
                    "prior_epochs": int(params["prior_epochs"]),
                    "prior_batch": int(params["prior_batch"]),
                    "prior_lr": float(params["prior_lr"]),
                    "use_ema": bool(params["use_ema"]),
                    "ema_decay": float(params["ema_decay"]),
                    "class_conditional_marginals": optional_flags.get(
                        "class_conditional_marginals", True
                    ),
                    "class_conditional_missingness": optional_flags.get(
                        "class_conditional_missingness", True
                    ),
                    "fold": int(fold_id),
                    "fold_seed": int(fold_seed),
                    "n_train": int(len(train_idx)),
                    "n_test": int(len(test_idx)),
                    "n_syn_requested": n_syn,
                    "status": "STARTED",
                    "error": "",
                }

                log(
                    f"[Run {counter + 1:03d}/{total_runs}] {variant_name} | "
                    f"{case_id} | fold={fold_id:02d}"
                )

                try:
                    set_seed(int(fold_seed))
                    cleanup_torch()
                    torch.cuda.reset_peak_memory_stats()

                    fit_start = time.perf_counter()
                    generator = fit_generator(
                        X_train=X_train,
                        y_train=y_train,
                        params=params,
                        optional_flags=optional_flags,
                        seed=int(fold_seed),
                        save_name=(
                            f"colon_compsens_{variant_name.replace('-', '_')}_"
                            f"{case_id.replace('-', '_')}_fold{fold_id}_cuda4"
                        ),
                    )
                    fit_seconds = time.perf_counter() - fit_start
                    peak_gpu_gib = torch.cuda.max_memory_allocated() / (1024 ** 3)

                    sample_start = time.perf_counter()
                    X_syn, _, y_syn = generator.sample(n=n_syn)
                    sample_seconds = time.perf_counter() - sample_start
                    X_syn, y_syn = sanitize_synthetic_training_data(
                        X_syn=X_syn,
                        y_syn=y_syn,
                        X_real_train=X_train,
                        y_real_train=y_train,
                        rng=fold_rng,
                    )

                    tstr_model = make_lr(int(fold_seed))
                    tstr_model.fit(X_syn, y_syn)
                    tstr_probabilities = predict_positive_probability(tstr_model, X_test)
                    tstr_metrics = binary_metrics(y_test, tstr_probabilities)

                    c2st_metrics = evaluate_c2st(
                        X_real_train=X_train,
                        X_syn=X_syn,
                        seed=int(fold_seed),
                    )

                    trtr_metrics = trtr_map[fold_id]
                    row.update(
                        {
                            "status": "COMPLETE",
                            "n_syn_actual": int(len(X_syn)),
                            "fit_seconds": float(fit_seconds),
                            "sample_seconds": float(sample_seconds),
                            "peak_gpu_gib": float(peak_gpu_gib),
                            "tstr_acc": tstr_metrics["acc"],
                            "tstr_auc": tstr_metrics["auc"],
                            "trtr_acc": float(trtr_metrics["trtr_acc"]),
                            "trtr_auc": float(trtr_metrics["trtr_auc"]),
                            "c2st_acc": c2st_metrics["acc"],
                            "c2st_auc": c2st_metrics["auc"],
                        }
                    )
                    log(
                        f"  TSTR ACC={row['tstr_acc']:.4f}, AUC={row['tstr_auc']:.4f} | "
                        f"C2ST ACC={row['c2st_acc']:.4f}, AUC={row['c2st_auc']:.4f} | "
                        f"fit={fit_seconds:.2f}s"
                    )
                    del generator, X_syn, y_syn, tstr_model

                except UnsupportedOptionalFlag as exc:
                    row.update(
                        {
                            "status": "SKIPPED_UNSUPPORTED",
                            "error": str(exc),
                        }
                    )
                    log(f"  SKIPPED: {exc}")

                except Exception as exc:
                    row.update(
                        {
                            "status": "FAIL",
                            "error": f"{type(exc).__name__}: {exc}",
                        }
                    )
                    log(f"  FAILED: {row['error']}")
                    traceback.print_exc()

                finally:
                    cleanup_torch()
                    results = append_and_save_result(results, row, fold_results_path)
                    counter += 1

    return results


# =============================================================================
# Summaries
# =============================================================================
def mean_std_text(mean: float, std: float, digits: int = 4) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def build_summary(fold_results: pd.DataFrame) -> pd.DataFrame:
    complete = fold_results[fold_results["status"] == "COMPLETE"].copy()
    if complete.empty:
        raise RuntimeError("No completed sensitivity runs are available.")

    metrics = [
        "tstr_acc",
        "tstr_auc",
        "trtr_acc",
        "trtr_auc",
        "c2st_acc",
        "c2st_auc",
        "fit_seconds",
        "sample_seconds",
        "peak_gpu_gib",
    ]
    keys = ["variant", "use_gobs", "case_id", "factor", "level"]
    grouped = complete.groupby(keys, as_index=False)
    summary = grouped[metrics].agg(["mean", "std"])
    summary.columns = [
        "_".join([str(part) for part in column if str(part)])
        for column in summary.columns.to_flat_index()
    ]
    summary = summary.rename(
        columns={
            "variant_": "variant",
            "use_gobs_": "use_gobs",
            "case_id_": "case_id",
            "factor_": "factor",
            "level_": "level",
        }
    )
    counts = grouped.size().rename(columns={"size": "n_folds"})
    summary = summary.merge(counts, on=keys, how="left")

    for metric in ["tstr_acc", "tstr_auc", "c2st_acc", "c2st_auc"]:
        summary[f"{metric}_formatted"] = summary.apply(
            lambda row: mean_std_text(
                float(row[f"{metric}_mean"]),
                float(row[f"{metric}_std"]),
            ),
            axis=1,
        )

    case_order = {case["case_id"]: i for i, case in enumerate(CASES)}
    summary["case_order"] = summary["case_id"].map(case_order)
    return summary.sort_values(["variant", "case_order"]).reset_index(drop=True)


def build_baseline_deltas(summary: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for variant, group in summary.groupby("variant"):
        baseline = group[group["case_id"] == "BASE"]
        if baseline.empty:
            continue
        base = baseline.iloc[0]
        for _, row in group.iterrows():
            rows.append(
                {
                    "variant": variant,
                    "case_id": row["case_id"],
                    "factor": row["factor"],
                    "level": row["level"],
                    "tstr_auc_mean": row["tstr_auc_mean"],
                    "delta_tstr_auc_vs_base": row["tstr_auc_mean"] - base["tstr_auc_mean"],
                    "tstr_acc_mean": row["tstr_acc_mean"],
                    "delta_tstr_acc_vs_base": row["tstr_acc_mean"] - base["tstr_acc_mean"],
                    "c2st_auc_mean": row["c2st_auc_mean"],
                    "delta_c2st_auc_vs_base": row["c2st_auc_mean"] - base["c2st_auc_mean"],
                    "fit_seconds_mean": row["fit_seconds_mean"],
                    "runtime_ratio_vs_base": row["fit_seconds_mean"] / max(base["fit_seconds_mean"], 1e-12),
                }
            )
    return pd.DataFrame(rows)


def build_factor_importance(delta_df: pd.DataFrame) -> pd.DataFrame:
    non_base = delta_df[delta_df["case_id"] != "BASE"].copy()
    rows: List[Dict[str, object]] = []
    for (variant, factor), group in non_base.groupby(["variant", "factor"]):
        rows.append(
            {
                "variant": variant,
                "factor": factor,
                "n_levels": int(len(group)),
                "worst_tstr_auc_drop": float(group["delta_tstr_auc_vs_base"].min()),
                "largest_abs_tstr_auc_change": float(
                    group["delta_tstr_auc_vs_base"].abs().max()
                ),
                "tstr_auc_range_across_tested_levels": float(
                    group["tstr_auc_mean"].max() - group["tstr_auc_mean"].min()
                ),
                "largest_abs_c2st_auc_change": float(
                    group["delta_c2st_auc_vs_base"].abs().max()
                ),
            }
        )
    output = pd.DataFrame(rows)
    if not output.empty:
        output = output.sort_values(
            ["variant", "largest_abs_tstr_auc_change"],
            ascending=[True, False],
        )
    return output


def save_plots(summary: pd.DataFrame, output_dir: Path) -> None:
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        log(f"Skipping plots because matplotlib is unavailable: {exc}")
        return

    for variant, group in summary.groupby("variant"):
        group = group.sort_values("case_order")
        labels = group["case_id"].tolist()
        positions = np.arange(len(labels))

        fig, ax = plt.subplots(figsize=(10.5, 5.2))
        ax.errorbar(
            positions,
            group["tstr_auc_mean"],
            yerr=group["tstr_auc_std"],
            marker="o",
            capsize=3,
        )
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_ylabel("TSTR AUC (mean ± std)")
        ax.set_title(f"Colon component sensitivity: {variant}")
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        fig.savefig(
            output_dir / f"{variant.replace('-', '_')}_tstr_auc_sensitivity.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(10.5, 5.2))
        ax.errorbar(
            positions,
            group["c2st_auc_mean"],
            yerr=group["c2st_auc_std"],
            marker="o",
            capsize=3,
        )
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_ylabel("C2ST AUC (mean ± std; lower is better)")
        ax.set_title(f"Colon C2ST sensitivity: {variant}")
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        fig.savefig(
            output_dir / f"{variant.replace('-', '_')}_c2st_auc_sensitivity.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.close(fig)


# =============================================================================
# Main
# =============================================================================
def main() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    set_seed(GLOBAL_SEED)
    device = validate_device()

    log("=" * 100)
    log("BSTabDiff Colon component/hyperparameter sensitivity")
    log("=" * 100)
    log("Physical GPU ID: 4")
    log(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
    log(f"Logical torch device: {device}")
    log(f"Visible GPU count: {torch.cuda.device_count()}")
    log(f"Visible GPU name: {torch.cuda.get_device_name(0)}")
    log(f"Fixed M: {FIXED_M}")
    log("Evaluator: Logistic Regression only")
    log(f"CV: {N_REPEATS} repeats × {N_SPLITS} folds")
    log(f"fit_block_subunit_generator signature: {FIT_SIGNATURE}")
    log(f"Resolved optional flags: {RESOLVED_OPTIONAL_FLAGS}")
    log(
        "Colon has no missing values; No-CC-Missingness is a sanity check, "
        "not a complete missingness sensitivity study."
    )

    X, y, feature_names = load_colon_dataset(DATA_FILE, TARGET_COL)
    log(f"Loaded Colon: X={X.shape}, y={y.shape}, classes={np.unique(y)}")

    config = {
        "data_file": DATA_FILE,
        "target_col": TARGET_COL,
        "shape": list(X.shape),
        "global_seed": GLOBAL_SEED,
        "cv_seed": CV_SEED,
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "fixed_M": FIXED_M,
        "physical_gpu_id": 4,
        "logical_device": TORCH_DEVICE_STR,
        "best_no_gobs": BEST_NO_GOBS,
        "best_with_gobs": BEST_WITH_GOBS,
        "matched_diffusion_overrides": MATCHED_DIFFUSION_OVERRIDES,
        "cases": CASES,
        "resolved_optional_flags": RESOLVED_OPTIONAL_FLAGS,
        "feature_count": len(feature_names),
    }
    with open(OUTPUT_DIR / "experiment_config.json", "w", encoding="utf-8") as handle:
        json.dump(config, handle, indent=2)

    rkf = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=CV_SEED,
    )
    splits = list(rkf.split(X, y))
    seed_rng = np.random.RandomState(CV_SEED)
    fold_seeds = seed_rng.randint(0, 2**31 - 1, size=len(splits), dtype=np.int64)

    trtr_path = OUTPUT_DIR / "colon_lr_trtr_5x5.csv"
    if RESUME and trtr_path.exists():
        trtr_by_fold = pd.read_csv(trtr_path)
        log(f"Loaded cached TRTR results: {trtr_path}")
    else:
        log("Computing Logistic Regression TRTR once for the fixed CV splits...")
        trtr_by_fold = evaluate_trtr_once(X, y, splits, fold_seeds)
        trtr_by_fold.to_csv(trtr_path, index=False)

    fold_results_path = OUTPUT_DIR / "colon_component_sensitivity_fold_results.csv"
    fold_results = run_sensitivity(
        X=X,
        y=y,
        splits=splits,
        fold_seeds=fold_seeds,
        trtr_by_fold=trtr_by_fold,
        fold_results_path=fold_results_path,
    )

    summary = build_summary(fold_results)
    summary_path = OUTPUT_DIR / "colon_component_sensitivity_summary.csv"
    summary.to_csv(summary_path, index=False)

    deltas = build_baseline_deltas(summary)
    deltas_path = OUTPUT_DIR / "colon_component_sensitivity_deltas.csv"
    deltas.to_csv(deltas_path, index=False)

    factor_importance = build_factor_importance(deltas)
    factor_path = OUTPUT_DIR / "colon_component_sensitivity_factor_ranking.csv"
    factor_importance.to_csv(factor_path, index=False)

    rebuttal_columns = [
        "variant",
        "case_id",
        "tstr_acc_formatted",
        "tstr_auc_formatted",
        "c2st_auc_formatted",
        "fit_seconds_mean",
    ]
    rebuttal_table = summary[rebuttal_columns].copy()
    rebuttal_table.to_csv(
        OUTPUT_DIR / "colon_component_sensitivity_rebuttal_table.csv", index=False
    )

    save_plots(summary, OUTPUT_DIR)

    log("\n=== Sensitivity summary ===")
    log(rebuttal_table.to_string(index=False))
    log("\n=== Factors ranked by largest TSTR-AUC sensitivity ===")
    log(factor_importance.to_string(index=False))
    log("\nSaved outputs in: " + str(OUTPUT_DIR.resolve()))

    return fold_results, summary, deltas, factor_importance


if __name__ == "__main__":
    main()

BSTabDiff Colon component/hyperparameter sensitivity
Physical GPU ID: 4
CUDA_VISIBLE_DEVICES: 4
Logical torch device: cuda:0
Visible GPU count: 1
Visible GPU name: NVIDIA RTX A6000
Fixed M: 8
Evaluator: Logistic Regression only
CV: 5 repeats × 5 folds
fit_block_subunit_generator signature: (X: 'np.ndarray', feature_specs: 'List[FeatureSpec]', y: 'Optional[np.ndarray]' = None, M: 'int' = 32, blocks: 'Optional[List[np.ndarray]]' = None, permute_features: 'bool' = False, prior_type: 'str' = 'diffusion', device: 'str' = 'cpu', seed: 'int' = 0, prior_epochs: 'int' = 1500, prior_batch: 'int' = 128, prior_lr: 'float' = 0.001, verbose_every: 'int' = 200, save_dir: 'Optional[str]' = None, save_name: 'str' = 'blocksubunit', save_best: 'bool' = True, use_ema: 'bool' = True, ema_decay: 'float' = 0.999, return_train_info: 'bool' = False, use_gobs: 'bool' = False, gobs_num_clusters: 'int' = 7, gobs_metric: 'str' = 'kl_divergence', gobs_bins: 'int' = 32, gobs_top_k: 'Optional[int]' = None, gobs_refin